# PneumoType: fast demo app
Loads the CNNs we already trained from scratch in the main **PneumoType** notebook (its saved output files `best.pt`, `best_robust.pt` and `fed_model.pt`) and launches the demo app in about a minute, with no retraining.
**Setup:** Add Input → *Your Work* → **PneumoType** (notebook output). Internet on. GPU optional.
*Prototype only. Not medical advice.*

In [ ]:
!pip install -q gradio

In [ ]:
import os, glob, time, io, urllib.request
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE, K = 224, 3
CLASSES = ["normal", "bacterial", "viral"]
FINAL_BONUS, TEMP, UNSURE_CUTOFF = 0.58, 0.95, 0.7     # from the main notebook (Version 4, chosen on validation)
TRACK = {0.5: 0.826, 0.6: 0.867, 0.7: 0.910, 0.8: 0.947, 0.9: 0.977}   # accuracy on test X-rays at/above each confidence
CONF_MIN = 1 - 0.8222   # split conformal (upgrades notebook): label is in the 95% set if its calibrated probability >= this
CHILD_TH, ADULT_TH = 0.42, 0.5   # federated pneumonia threshold per hospital (upgrades notebook)

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found: add the PneumoType notebook output as an input"
    return hits[0]

def block(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                         nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True), nn.MaxPool2d(2))
class OurCNN(nn.Module):
    def __init__(self):
        super().__init__(); ch = [1, 32, 64, 128, 256, 512]
        self.features = nn.Sequential(*[block(a, b) for a, b in zip(ch[:-1], ch[1:])])
class PneumoType(nn.Module):
    def __init__(self, n_out=K):
        super().__init__(); self.backbone = OurCNN(); self.drop = nn.Dropout(0.3); self.head = nn.Linear(512, n_out)
    def forward(self, x):
        f = self.backbone.features((x - 0.5) / 0.25)
        return self.head(self.drop(F.adaptive_avg_pool2d(F.relu(f), 1).flatten(1)))

def load(path, n_out):
    m = PneumoType(n_out).to(device); m.load_state_dict(torch.load(path, map_location=device)); return m.eval()
model, model2 = load(find("best.pt"), 3), load(find("best_robust.pt"), 3)
FED_OK = len(glob.glob("/kaggle/input/**/fed_model.pt", recursive=True)) > 0
fed_model = load(find("fed_model.pt"), 2) if FED_OK else None
print("loaded CNN 1, CNN 2", "+ federated model" if FED_OK else "")

eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor()])
tta_tf = T.Compose([T.Resize((248, 248)), T.CenterCrop(IMG_SIZE), T.ToTensor()])

In [ ]:
@torch.no_grad()
def probs_of(m, pil, tf=eval_tf):
    return F.softmax(m(tf(pil.convert("L")).unsqueeze(0).to(device)).float(), 1)[0].cpu().numpy()

def final_probs(pil):
    ps = [probs_of(m, pil, tf) for m in (model, model2) for tf in (eval_tf, tta_tf)]
    p = np.mean(ps, 0); z = np.log(np.clip(p, 1e-8, 1)) / TEMP; z = np.exp(z - z.max()); return z / z.sum()

def gradcam(pil, m, target):
    x = eval_tf(pil.convert("L")).unsqueeze(0).to(device); acts = {}
    def hook(mod, i, o): acts["a"] = o; o.register_hook(lambda g: acts.__setitem__("g", g))
    h = m.backbone.features.register_forward_hook(hook); logits = m(x); h.remove()
    m.zero_grad(); logits[0, target].backward()
    cam = F.relu((acts["g"].mean(dim=(2, 3), keepdim=True) * acts["a"]).sum(1)).squeeze().detach().cpu().numpy()
    cam = np.array(Image.fromarray(cam).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))
    return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

def input_checks(pil, lung):
    rgb = np.asarray(pil.convert("RGB")).astype(float); g = np.asarray(pil.convert("L")).astype(float)
    colour = float(np.abs(rgb - rgb.mean(2, keepdims=True)).mean()); w, h = pil.size
    out = []
    if colour > 8: out.append(("FAIL", "Image has colour: this may not be a chest X-ray. Result not reliable."))
    else: out.append(("PASS", "Greyscale image, consistent with a radiograph"))
    if min(w, h) < 256: out.append(("WARN", f"Low resolution ({w}x{h}); robust CNN weighted in, but treat with caution"))
    else: out.append(("PASS", f"Resolution OK ({w}x{h})"))
    if g.std() < 35: out.append(("WARN", f"Low contrast (pixel SD {g.std():.0f}); closer to the 'different clinic' images we stress-tested"))
    else: out.append(("PASS", "Contrast OK"))
    if lung < 0.5: out.append(("WARN", f"Only {lung:.0%} of the model's attention is inside the lung region: possible shortcut, needs a human read"))
    else: out.append(("PASS", f"{lung:.0%} of the model's attention is inside the lung region"))
    return out

def make_report(cls, cset, set_txt, probs, conf, agree, unsure, checks, every, fed_child, fed_adult, ms):
    failed = any(c[0] == "FAIL" for c in checks)
    if failed or unsure or len(cset) == 3:
        level, why = "REFER", "the system is not confident enough to triage this X-ray"
    elif cset == ["bacterial"] or (cls == 1 and "bacterial" in cset):
        level, why = "URGENT", "findings suggest bacterial pneumonia"
    elif set(cset) == {"bacterial", "viral"} or (len(cset) == 2 and "normal" not in cset):
        level, why = "REVIEW TODAY", "pneumonia likely; bacterial vs viral cannot be separated from the X-ray alone"
    elif cset == ["viral"]:
        level, why = "REVIEW", "findings suggest viral pneumonia"
    elif cset == ["normal"]:
        level, why = "ROUTINE", "no radiographic pneumonia detected"
    else:
        level, why = "REVIEW", "normal and pneumonia both plausible"
    steps = {
        "URGENT": ["Clinician review today", "Bacterial pneumonia is likely: decide on antibiotics per local guidelines", "Check oxygen saturation and signs of severe illness"],
        "REVIEW TODAY": ["Clinician review today", "Bloodwork such as CRP or procalcitonin can help decide whether antibiotics are needed", "Re-assess if breathing worsens"],
        "REVIEW": ["Clinician review", "Viral pattern: supportive care is usual; safety-net advice for worsening symptoms", "Consider bacterial co-infection if the child is very unwell"],
        "ROUTINE": ["No pneumonia seen on this X-ray", "Clinical judgement still applies: early pneumonia can be invisible on X-ray"],
        "REFER": ["Do not rely on this result", "Send for radiologist read"],
    }[level]
    icon = {"PASS": "✅", "WARN": "⚠️", "FAIL": "❌"}
    lines = [f"## Triage: **{level}**", f"*Reason:* {why}.", "",
             "**Findings**",
             f"- Conformal 95% answer: **{set_txt}**",
             f"- Most likely: {CLASSES[int(probs.argmax())]} ({conf:.0%}); bacterial {probs[1]:.0%} · viral {probs[2]:.0%} · normal {probs[0]:.0%}",
             f"- Two independent networks agree: {'yes' if agree else 'no'}"]
    if every: lines.append(f"- Federated all-ages check: pneumonia {every['pneumonia']:.0%} (children's-hospital threshold: {fed_child}; adult: {fed_adult})")
    lines += ["", "**Suggested next steps**"] + [f"{i+1}. {t}" for i, t in enumerate(steps)]
    lines += ["", "**Automatic safety checks**"] + [f"- {icon[a]} {b}" for a, b in checks]
    lines += ["", f"*Generated in {ms:.0f} ms by PneumoType. Prototype decision support, not medical advice; a clinician makes the decision.*"]
    md = "\n".join(lines)
    path = "/kaggle/working/pneumotype_report.md" if os.path.isdir("/kaggle/working") else "pneumotype_report.md"
    open(path, "w").write("# PneumoType triage report\n\n" + md.replace("## ", "### "))
    return md, path

def analyze(img):
    if img is None: return "### Upload or pick an X-ray", None, None, None, None, "", None
    t0 = time.time(); pil = Image.fromarray(img) if not isinstance(img, Image.Image) else img
    p1, p2 = probs_of(model, pil), probs_of(model2, pil)
    probs = final_probs(pil); adj = probs.copy(); adj[1] += FINAL_BONUS; cls = int(adj.argmax())
    conf = float(probs.max()); ent = float(-(probs * np.log(probs + 1e-9)).sum() / np.log(K))
    agree = int(p1.argmax()) == int(p2.argmax())
    cam = gradcam(pil, model2, cls)
    base = np.array(pil.convert("RGB").resize((IMG_SIZE, IMG_SIZE))) / 255.0
    overlay = ((0.65 * base + 0.35 * plt.cm.jet(cam)[..., :3]) * 255).astype(np.uint8)
    lung = float(cam[int(.12*IMG_SIZE):int(.92*IMG_SIZE), int(.15*IMG_SIZE):int(.85*IMG_SIZE)].sum() / (cam.sum() + 1e-8))
    every = None; fed_child = fed_adult = "n/a"
    if FED_OK:
        fp = probs_of(fed_model, pil); every = {"normal": float(fp[0]), "pneumonia": float(fp[1])}
        fed_child = "pneumonia" if fp[1] >= CHILD_TH else "normal"; fed_adult = "pneumonia" if fp[1] >= ADULT_TH else "normal"
    cset = [c for c, p in zip(CLASSES, probs) if p >= CONF_MIN] or [CLASSES[int(probs.argmax())]]
    SET_TEXT = {("normal",): "Normal", ("bacterial",): "Bacterial pneumonia", ("viral",): "Viral pneumonia",
                ("bacterial", "viral"): "Pneumonia, type unclear (bacterial or viral)",
                ("normal", "bacterial", "viral"): "Can't narrow it down: refer"}
    set_txt = SET_TEXT.get(tuple(cset), " or ".join(cset))
    if conf < UNSURE_CUTOFF or not agree:
        why = "the two networks disagree" if not agree else f"confidence {conf:.0%} is below {UNSURE_CUTOFF:.0%}"
        head = f"## UNSURE: refer to a doctor / radiologist\n*Why:* {why}."
    else:
        head = {0: "## Normal", 1: "## Bacterial pneumonia: likely needs antibiotics, see a doctor now", 2: "## Viral pneumonia: see a doctor"}[cls]
    head += (f"\n\n**95% guaranteed answer (conformal prediction):** {set_txt}  \n"
             f"*The true diagnosis is inside this answer 95% of the time (96.4% on 799 unseen X-rays). "
             f"One label = confident; more labels = the model honestly can't tell.*")
    level = max([c for c in TRACK if c <= conf], default=None)
    if level is not None:
        head += f"\n\nOn held-out test X-rays where PneumoType was at least {level:.0%} confident, it was right **{TRACK[level]:.0%}** of the time."
    head += "\n\n*Prototype only. Not medical advice.*"
    stats = pd.DataFrame([
        ["CNN 1 says", f"{CLASSES[int(p1.argmax())]} ({p1.max():.0%})"],
        ["CNN 2 (clinic-robust) says", f"{CLASSES[int(p2.argmax())]} ({p2.max():.0%})"],
        ["Networks agree?", "yes" if agree else "NO: flagged as unsure"],
        ["Ensemble + TTA (calibrated) top class", f"{CLASSES[int(probs.argmax())]} ({conf:.0%})"],
        ["Conformal 95% set", " / ".join(cset) + f"  (labels with prob >= {CONF_MIN:.2f})"],
        ["Decision after safety tuning", CLASSES[cls]],
        ["Uncertainty (normalized entropy)", f"{ent:.2f}  (0 = certain, 1 = coin flip)"],
        ["Safety bonus applied to bacterial", f"+{FINAL_BONUS:.2f}"],
        ["Heatmap focus inside lung region", f"{lung:.0%}"],
        ["Pneumonia check, anyone (federated)", f"{every['pneumonia']:.0%}" if every else "n/a"],
        ["Federated verdict at children's / adult hospital threshold", f"{fed_child} (>= {CHILD_TH}) / {fed_adult} (>= {ADULT_TH})" if every else "n/a"],
        ["Analysis time", f"{(time.time() - t0) * 1000:.0f} ms"],
    ], columns=["Signal", "Value"])
    unsure = conf < UNSURE_CUTOFF or not agree
    report, rpath = make_report(cls, cset, set_txt, probs, conf, agree, unsure, input_checks(pil, lung), every, fed_child, fed_adult, (time.time() - t0) * 1000)
    return head, {c: float(p) for c, p in zip(CLASSES, probs)}, every, overlay, stats, report, rpath

In [ ]:
import gradio as gr
# example X-rays from the children's test set, if the dataset is attached
ex = sorted(glob.glob("/kaggle/input/**/test/PNEUMONIA/*.jpeg", recursive=True))[:4] + sorted(glob.glob("/kaggle/input/**/test/NORMAL/*.jpeg", recursive=True))[:2]
def fetch(url):
    try: return np.array(Image.open(io.BytesIO(urllib.request.urlopen(url, timeout=20).read())).convert("RGB"))
    except Exception: return None
REPO = "https://raw.githubusercontent.com/Riyadua777/pneumotype/main/figures/"
SYSTEM_IMG, RESULTS_IMG = fetch(REPO + "system.png"), fetch(REPO + "results.png")

METRICS = '''### Children: normal / bacterial / viral (held-out test, 799 X-rays)
| | Simple baseline | **PneumoType** | **+ safety tuning** |
|---|---|---|---|
| Accuracy | 77.1% | **84.4%** | 81.6% |
| Macro recall | 77.2% | **84.2%** | 77.5% |
| Pneumonia caught | 95.3% | 97.2% | **99.3%** |
| Dangerous misses (bacterial) | 92 | 62 | **18** |
| False alarms (of 222 normal) | 25 | **7** | 18 |

**95% confidence intervals (bootstrapped by patient):** accuracy 84.4% [81.6–87.0], pneumonia caught 97.2% [95.8–98.5]
**Conformal prediction (95% target):** true diagnosis inside the answer **96.4%** of the time; single-label answers right **95.0%** of the time
**Calibration error (ECE):** 0.044 → 0.037 after temperature scaling · **When ≥90% confident:** right 97.7% of the time
**Robustness:** first CNN fell to 0.66 on low-contrast X-rays; the clinic-robust CNN stays 0.83 on every kind of image
**Shortcut test:** lungs blacked out → chance (0.34), so it reads the lungs, not the borders
**Federated learning (children + adults, no X-rays shared):** balanced accuracy 88.4% on children, 83.1% on adults (best worst-case of any private setup); a children's-hospital threshold raises pneumonia caught from 82% to 87%'''

with gr.Blocks(title="PneumoType", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# PneumoType: chest X-ray pneumonia triage\nTwo CNNs trained **from scratch**, ensembled, calibrated and safety-tuned, plus a federated model trained across a children's and an adult hospital. *Prototype for a children's clinic, not medical advice.*")
    with gr.Tab("Diagnose"):
        with gr.Row():
            with gr.Column(scale=1):
                inp = gr.Image(label="Chest X-ray", type="numpy")
                btn = gr.Button("Analyze X-ray", variant="primary")
                if ex: gr.Examples(ex, inputs=inp, label="Example X-rays from the test set (click one)")
            with gr.Column(scale=2):
                verdict = gr.Markdown()
                with gr.Row():
                    type_lbl = gr.Label(label="Pneumonia type (children model, calibrated)", num_top_classes=3)
                    every_lbl = gr.Label(label="Pneumonia check for anyone (federated)")
                with gr.Row():
                    cam_out = gr.Image(label="Grad-CAM: where the model looked")
                    stats_out = gr.Dataframe(label="Under the hood", wrap=True)
                with gr.Row():
                    report_out = gr.Markdown(label="Clinical triage report")
                report_file = gr.File(label="Download triage report")
        outs = [verdict, type_lbl, every_lbl, cam_out, stats_out, report_out, report_file]
        btn.click(analyze, inp, outs)
        inp.change(analyze, inp, outs)
    with gr.Tab("How good is it?"):
        gr.Markdown(METRICS)
        if RESULTS_IMG is not None: gr.Image(RESULTS_IMG, label="Results dashboard (held-out test data)")
    with gr.Tab("How it works"):
        if SYSTEM_IMG is not None: gr.Image(SYSTEM_IMG, label="The PneumoType system")
        gr.Markdown('''1. **Two CNNs trained from scratch** (5 conv blocks, ~4.7M parameters each). CNN 2 is trained on blurred, low-contrast and low-resolution X-rays.
2. **Ensemble + test-time augmentation**, chosen on validation data.
3. **Calibration** so confidence matches accuracy (ECE 0.044 → 0.037).
4. **Safety tuning**: +0.58 to bacterial so missed bacterial cases drop from 62 to 18.
5. **Unsure mode**: below 70% confidence, or if the two networks disagree, it refers to a doctor.
6. **Federated learning (FedAvg)**: hospitals share only model weights, never X-rays.
7. **Shortcut checks**: lungs blacked out to make sure it reads the lungs.
8. **Conformal prediction**: a 95%-guaranteed set of possible diagnoses, calibrated on validation data.''')
demo.launch(share=True)

In [ ]:
# keep the session busy so Kaggle doesn't shut the demo down while you record
while True: time.sleep(60)